In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="ylacombe/google-argentinian-spanish", 
                  repo_type="dataset", local_dir="./google-argentinian-spanish")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [00:02<00:00,  2.80it/s]


'/home/ubuntu/google-argentinian-spanish'

In [3]:
files = glob('google-argentinian-spanish/*/*.parquet')
len(files)

6

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1/1 [01:22<00:00, 82.39s/it]


In [7]:
len(data)

5739

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'google-argentinian-spanish_male_audio/google-argentinian-spanish-male-train-00001-of-00002-f6f0bfbdc6bb1de2_0.mp3',
 'text': 'Estoy aburrido con la música de mi teléfono, quiero que me recomiendes algo nuevo.',
 'speaker': 'google-argentinian-spanish_male_audio_610'}

In [9]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'google-argentinian-spanish')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 411.85ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  155kB /  155kB, 15.2kB/s  
Processing Files (1 / 1): 100%|██████████|  155kB /  155kB, 15.5kB/s  
New Data Upload: 100%|██████████|  155kB /  155kB, 15.5kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.94s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/8cd3b599ac945d95eb2a9277843a25992a6e9b0c', commit_message='Upload dataset', commit_description='', oid='8cd3b599ac945d95eb2a9277843a25992a6e9b0c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('google-argentinian-spanish-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [12]:
folders = glob('google-argentinian-spanish_*_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

google-argentinian-spanish_male_audio_neucodec
google-argentinian-spanish_female_audio
google-argentinian-spanish_male_audio
google-argentinian-spanish_female_audio_neucodec


In [14]:
# from huggingface_hub import HfApi
# api = HfApi()

# for f in glob('google-argentinian-spanish_*_audio*.zip'):
#     api.upload_file(
#         path_or_fileobj=f,
#         path_in_repo=f,
#         repo_id="malaysia-ai/Multilingual-TTS",
#         repo_type="dataset",
#     )